# Laboratorio 1 · Entorno y primeros casos comprobados

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udd-ia-generativa-2026/curso-ia/blob/lab01/notebooks/lab01.ipynb)

**Formativo, sin nota.** Trabajo individual en Google Colab: instalas el código del curso, ejecutas un caso común en dos modelos y entregas tu evidencia en Canvas.

Al terminar tendrás: una llamada a un modelo escrita por ti, casos ejecutados en dos modelos, una revisión de cada respuesta y una tabla con tokens, latencia y costo.

Antes de empezar, abre **Secretos** (el ícono de llave del panel izquierdo) y agrega tu clave de Gemini como `GOOGLE_API_KEY` o `GEMINI_API_KEY` (el nombre que usa el botón de Colab) y, si el docente te los entregó, `CURSO_API_KEY` y `CURSO_BASE_URL` del servidor del curso, con acceso para este notebook. El servidor del curso es un respaldo que solo funciona durante las clases de los viernes; Gemini es la ruta principal. En vez de los secretos del servidor también puedes descomentar las dos líneas de la celda de configuración y pegar la clave del día. No escribas tu clave de Gemini en las celdas ni imprimas `os.environ`.

Al instalar, pip puede avisar un conflicto de `google-auth` con `google-colab`: es esperado y no afecta al laboratorio. Si el docente publica una versión nueva durante la clase, usa **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar desde la primera celda.

In [4]:
import os
import sys

# En Colab se instala el código del curso; en un repositorio con entorno propio ya está instalado.
if "COLAB_RELEASE_TAG" in os.environ:
    %pip install -q "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@lab01"
    # Reinstala solo el paquete del curso, por si quedó una copia anterior con la misma versión.
    %pip install -q --force-reinstall --no-deps "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@lab01"
    if "curso_ia" in sys.modules:
        print("El paquete ya estaba cargado: usa Entorno de ejecución → Reiniciar sesión y vuelve a ejecutar desde aquí.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
El paquete ya estaba cargado: usa Entorno de ejecución → Reiniciar sesión y vuelve a ejecutar desde aquí.


In [5]:
import os
from pathlib import Path

import pandas as pd
from pydantic_ai import Agent

from curso_ia import (
    apply_review, blocked, create_model, lab_cases, load_config, parse_cases, retry_failed, review_template,
    run_case, save_run, status,
)

# Modelo de comparación: texto libre con la forma proveedor:modelo. Opciones recomendadas:
#   "google:gemma-4-26b-a4b-it"  Gemma 4, de pesos abiertos, gratis con tu clave de Gemini; puede tardar.
#   "curso:course-chat"          servidor del curso con Gemma 4 12B, solo durante las clases de los viernes;
#                                requiere los secretos CURSO_API_KEY y CURSO_BASE_URL.
COMPARISON_MODEL = "google:gemma-4-26b-a4b-it"
os.environ["COMPARISON_MODEL"] = COMPARISON_MODEL

# Servidor del curso sin Secretos: descomenta y pega la clave del día que entrega el docente.
# os.environ["CURSO_BASE_URL"] = "https://tofu.tail07b663.ts.net/v1"
# os.environ["CURSO_API_KEY"] = ""

config = load_config()
lab = Path("lab01")  # aquí quedan la evidencia y el archivo para Canvas
if config.mode == "simulated":
    print("MODO SIMULADO: respuestas simuladas; no acredita acceso a ningún modelo.")
for row in status(config):
    state = "listo" if row["ready"] else "falta: " + ", ".join(row["missing"])
    print(f"{row['role']:<11} {row['model']:<32} {state}")


generation  google:gemini-3.5-flash-lite     listo
comparison  google:gemma-4-26b-a4b-it        listo


La lista muestra qué modelo atiende cada rol (`proveedor:modelo`) y, si falta algo, el **nombre** de la variable; nunca su valor. `generation` usa Gemini y `comparison` usa el modelo que escribiste en `COMPARISON_MODEL`, en la celda anterior. Por defecto es Gemma 4, un modelo de pesos abiertos de Google que la API de Gemini sirve gratis, así que la comparación es entre un modelo comercial y uno abierto. Durante la clase puedes usar en su lugar el servidor del curso con `curso:course-chat`; fuera de clase ese servidor no responde. Si cambias `COMPARISON_MODEL`, vuelve a ejecutar esa celda. Con una clave propia también puedes cambiar `generation` con la variable `GENERATION_MODEL`.

## 1. Tu primera llamada

Un `Agent` de PydanticAI recibe un modelo y una instrucción; `run` envía la consulta y devuelve la respuesta (`output`) y el consumo (`usage`), con los tokens de entrada y de salida que determinan el costo. La celda usa el primer perfil que esté listo.

`run` es asíncrona: en un notebook se escribe `await agent.run(...)` directo en la celda, porque Jupyter y Colab ya tienen un bucle de eventos (asyncio) en marcha. El editor de Colab puede subrayar `await` con el aviso «allowed only within async function»: es un aviso del editor y la celda se ejecuta igual. `run_sync` no sirve aquí, justamente porque ese bucle ya está corriendo.

In [6]:
ready = [row["role"] for row in status(config) if row["ready"]]
role = ready[0] if ready else "generation"  # el primer perfil listo; si ninguno, muestra qué falta
print("Perfil:", role, config.models[role])
model = create_model(config.models[role], config)
agent = Agent(model, instructions="Responde en una sola frase.")
result = await agent.run("¿Qué es un token en un modelo de lenguaje?")
print(result.output)
print(result.usage)


Perfil: generation google:gemini-3.5-flash-lite
Un token es la unidad básica de texto, que puede ser una palabra, parte de ella o un signo de puntuación, en la que el modelo descompone la información para procesarla.
RunUsage(cost=Decimal('0.0001035'), details={'text_prompt_tokens': 20}, input_text_tokens=20, input_tokens=20, output_tokens=39, requests=1)


## 2. Los casos

El caso común del curso viene en el paquete: la regla de tarifas por duelo del caso Air Canada en cuatro consultas, cada una con su respuesta esperada. Ejecútalo sin cambios.

Mínimo para hoy: el caso común en ambos perfiles; si el servidor del curso no está disponible, basta con Gemini y el bloqueo registrado. Extensión: en la celda siguiente, reemplaza el ejemplo de `MY_TASK` por dos o tres casos de una tarea que te interese (una instrucción y, por caso, una entrada y la respuesta que esperas), cambia `USE_MY_TASK` a `True` y vuelve a ejecutar desde aquí. Usa datos públicos o sintéticos.

In [7]:
MY_TASK = {
    "instruction": 'Clasifica el mensaje como "reclamo", "consulta" o "felicitación". Responde solo con la categoría.',
    "cases": [
        {"id": "ejemplo-1", "input": "¿A qué hora abre la sucursal de Providencia el sábado?", "expected": "consulta"},
        {"id": "ejemplo-2", "input": "Me cobraron dos veces la misma compra y nadie responde.", "expected": "reclamo"},
    ],
}
USE_MY_TASK = False  # extensión: True para ejecutar tus casos

cases = parse_cases(MY_TASK, "my-task") if USE_MY_TASK else lab_cases(1)
if cases.is_example:
    print("Aún usas los casos de ejemplo: reemplázalos por los de tu tarea.")
print("Instrucción:", cases.instruction)
pd.DataFrame([vars(case) for case in cases.cases])


Instrucción: Explica en qué información basas tu respuesta. Si no puedes confirmar una regla con la información disponible, indícalo.


,id,input,expected
0,sin-politica,Ya completé mi viaje. ¿Puedo solicitar ahora q...,No afirma una regla: indica que necesita la po...
1,con-politica,Documento para el ejercicio — síntesis didácti...,"No: según el documento, las solicitudes poster..."
2,antes-del-viaje,Documento para el ejercicio — síntesis didácti...,El documento solo excluye las solicitudes post...
3,dato-ausente,Documento para el ejercicio — síntesis didácti...,El documento no indica el porcentaje; hace fal...


## 3. Ejecutar en ambos perfiles

Cada caso se envía por separado, sin mensajes anteriores. Si un perfil no está configurado o falla, se registra el bloqueo y se sigue con el otro.

Si un caso queda en `error`, las columnas `error_type` y `http_status` dicen por qué; un 429 es el límite de solicitudes por minuto del nivel gratuito de Gemini: espera un minuto y vuelve a ejecutar. Un 503 indica alta demanda en Google: espera unos minutos y vuelve a ejecutar. Evita cambiar a un modelo Flash sin «Lite»: en el nivel gratuito solo admiten 20 solicitudes al día.

`SETTINGS` ajusta cómo responde el modelo, por ejemplo la temperatura o el máximo de tokens; ver [ajustes de modelo en PydanticAI](https://ai.pydantic.dev/agents/#model-run-settings) y la [lista de campos de `ModelSettings`](https://ai.pydantic.dev/api/settings/) (inglés).

Extensión: sube `REPETITIONS` a 2 o 3 para ver cuánto varían la respuesta, los tokens y la latencia.

In [8]:
from tqdm.auto import tqdm

REPETITIONS = 1
SETTINGS = {}  # por ejemplo {"temperature": 0.2}; vacío usa el valor de cada proveedor

rows = []
progress = tqdm(total=len(config.models) * len(cases.cases) * REPETITIONS, desc="Consultas")
for profile, ref in config.models.items():
    try:
        agent = Agent(create_model(ref, config), instructions=cases.instruction, model_settings=SETTINGS)
    except Exception as error:
        rows += [blocked(profile, ref, case, config, 1, error) for case in cases.cases]
        progress.update(len(cases.cases) * REPETITIONS)
        continue
    for case in cases.cases:
        for repetition in range(1, REPETITIONS + 1):
            progress.set_postfix_str(f"{profile} · {case.id}")
            rows.append(await run_case(agent, case, profile=profile, ref=ref, config=config, repetition=repetition))
            progress.update()
progress.close()

columns = [
    "profile", "case", "repetition", "status", "input_tokens", "output_tokens", "latency_ms",
    "estimated_cost_usd", "error_type", "http_status",
]
pd.DataFrame(rows).reindex(columns=columns)


Consultas:   0%|          | 0/8 [00:00<?, ?it/s]

,profile,case,repetition,status,input_tokens,output_tokens,latency_ms,estimated_cost_usd,error_type,http_status
0,generation,sin-politica,1,ok,47.0,252.0,1977,0.000644,NaN,NaN
1,generation,con-politica,1,ok,102.0,122.0,917,0.000336,NaN,NaN
2,generation,antes-del-viaje,1,ok,102.0,118.0,1001,0.000326,NaN,NaN
3,generation,dato-ausente,1,ok,98.0,45.0,780,0.000142,NaN,NaN
4,comparison,sin-politica,1,ok,47.0,1413.0,30418,0.000000,NaN,NaN
5,comparison,con-politica,1,ok,102.0,681.0,14707,0.000000,NaN,NaN
6,comparison,antes-del-viaje,1,ok,102.0,984.0,21274,0.000000,NaN,NaN
7,comparison,dato-ausente,1,error,NaN,NaN,10020,NaN,ModelHTTPError,500.0


### Reintentar las que fallaron

Si alguna consulta quedó en `error` (por ejemplo un 500, un 503 o un `TimeoutError`), la celda siguiente la vuelve a enviar hasta `MAX_RETRIES` veces, con una pausa de `WAIT_SECONDS` antes de cada intento para no chocar con el límite por minuto. Las consultas bloqueadas por falta de configuración no se reintentan. Si no hubo errores, la celda no envía nada.

La columna `attempts` muestra cuántos intentos usó cada consulta y queda en la evidencia: también es una medición de la confiabilidad de cada modelo. Puedes ejecutar la celda otra vez si aún quedan errores.

In [9]:
MAX_RETRIES = 2  # reintentos por consulta fallida
WAIT_SECONDS = 20  # pausa antes de cada reintento

rows = await retry_failed(rows, cases, config, settings=SETTINGS, max_retries=MAX_RETRIES, wait_s=WAIT_SECONDS)
pd.DataFrame(rows).reindex(columns=columns + ["attempts"])

Reintento 1 de 2: comparison · dato-ausente


,profile,case,repetition,status,input_tokens,output_tokens,latency_ms,estimated_cost_usd,error_type,http_status,attempts
0,generation,sin-politica,1,ok,47,252,1977,0.000644,NaN,NaN,1
1,generation,con-politica,1,ok,102,122,917,0.000336,NaN,NaN,1
2,generation,antes-del-viaje,1,ok,102,118,1001,0.000326,NaN,NaN,1
3,generation,dato-ausente,1,ok,98,45,780,0.000142,NaN,NaN,1
4,comparison,sin-politica,1,ok,47,1413,30418,0.000000,NaN,NaN,1
5,comparison,con-politica,1,ok,102,681,14707,0.000000,NaN,NaN,1
6,comparison,antes-del-viaje,1,ok,102,984,21274,0.000000,NaN,NaN,1
7,comparison,dato-ausente,1,ok,98,597,13127,0.000000,NaN,NaN,2


### Leer las respuestas

La tabla anterior trae las mediciones. Aquí se ven las respuestas completas, agrupadas por caso: primero la respuesta esperada y luego lo que respondió cada perfil. Léelas antes de revisarlas en la sección 5.

In [10]:
from IPython.display import Markdown, display

expected = {case.id: case.expected for case in cases.cases}
display(Markdown(f"**Instrucción:** {cases.instruction}"))

for row in rows:
    if row["status"] != "ok" and row["case"] == cases.cases[0].id:
        reason = ", ".join(row.get("missing", [])) or row.get("error_type", row["status"])
        display(Markdown(f"> **{row['profile']}** ({row['model']}) no respondió: {reason}."))

for case in cases.cases:
    display(Markdown(f"---\n### {case.id}\n**Esperada:** {expected[case.id]}"))
    for row in rows:
        if row["case"] != case.id or row["status"] != "ok":
            continue
        cost = f" · {row['estimated_cost_usd']} USD" if row.get("estimated_cost_usd") is not None else ""
        display(Markdown(
            f"#### {row['profile']} · {row['model']} (repetición {row['repetition']})\n\n{row['response']}\n\n"
            f"<sub>{row['input_tokens']} tokens de entrada · {row['output_tokens']} de salida · "
            f"{row['latency_ms']} ms{cost}</sub>"
        ))


**Instrucción:** Explica en qué información basas tu respuesta. Si no puedes confirmar una regla con la información disponible, indícalo.

---
### sin-politica
**Esperada:** No afirma una regla: indica que necesita la política de la aerolínea para confirmarlo.

#### generation · google:gemini-3.5-flash-lite (repetición 1)

Basándome en las políticas habituales de las aerolíneas y empresas de transporte, **no puedo confirmar una regla general**, ya que esto depende enteramente de la política específica de la compañía con la que realizaste tu viaje. 

Sin embargo, en la mayoría de los casos de la industria:
* **Restricción de tiempo:** La gran mayoría de las aerolíneas y empresas exigen que la tarifa por duelo (o el reembolso/ajuste posterior) se solicite **antes** de realizar el viaje o, como máximo, durante los primeros días posteriores al suceso, requiriendo la presentación de documentación específica (como certificado de defunción y prueba de parentesco).
* **Política retroactiva:** Son muy pocas las empresas que permiten aplicar descuentos o tarifas especiales de forma retroactiva una vez que el servicio ya ha sido completado y utilizado.

**Recomendación:** No cuento con información sobre la empresa específica con la que viajaste ni con los detalles de tu reserva. Te sugiero que te pongas en contacto directamente con el servicio de atención al cliente de la compañía lo antes posible para explicarles tu situación y consultar si hacen excepciones o tienen algún proceso para reembolsos por duelo posteriores al viaje.

<sub>47 tokens de entrada · 252 de salida · 1977 ms · 0.0006441 USD</sub>

#### comparison · google:gemma-4-26b-a4b-it (repetición 1)

**No puedo confirmar con certeza si puedes solicitarla, ya que no conozco la aerolínea o empresa de transporte con la que viajaste ni sus políticas específicas.**

Sin embargo, para ayudarte, te explico en qué me baso para darte una orientación general:

### Información en la que baso mi respuesta
Mi respuesta se basa en las **prácticas estándar de la industria de viajes y aviación comercial**. Aunque cada empresa tiene su propio reglamento, la mayoría sigue un protocolo similar para las tarifas especiales.

### La regla general (No confirmada para tu caso específico)
Basándome en el funcionamiento habitual de estas tarifas, la respuesta más probable es **no**, pero aquí están los detalles:

1.  **Solicitud previa a la compra:** La gran mayoría de las aerolíneas requieren que la tarifa por duelo se gestione y se apruebe **antes o en el momento de la compra del boleto**. Estas tarifas no son un "descuento" que se aplica a un precio ya pagado, sino una categoría de tarifa distinta que debe ser validada por un agente de ventas antes de emitir el ticket.
2.  **Falta de retroactividad:** Es extremadamente raro que las empresas apliquen tarifas especiales de forma retroactiva una vez que el servicio ya fue prestado y el viaje completado. Una vez que el boleto ha sido utilizado, el contrato de transporte se considera ejecutado al precio que se aceptó en su momento.
3.  **Validación de documentos:** Estas tarifas suelen requerir la presentación de documentos (como el certificado de defunción) en el momento de la reserva para justificar el precio especial.

### ¿Qué te sugiero hacer?
Debido a que **no puedo confirmar la regla exacta de tu proveedor**, te recomiendo lo siguiente:

*   **Contacta directamente al servicio al cliente:** Llama o escribe a la aerolínea. Explica tu situación con empatía y honestidad. Aunque la política general sea no aplicar tarifas retroactivas, en casos muy excepcionales y dependiendo de la política de "gestión de clientes" de la empresa, podrían ofrecerte alguna compensación o crédito de viaje (aunque no sea la tarifa por duelo propiamente dicha).
*   **Ten la documentación a mano:** Si decides reclamar, ten listo el certificado de defunción y el comprobante de tu parentesco, ya que será lo primero que te pedirán.

**Resumen de mi limitación:** No tengo acceso a tu itinerario, a tu cuenta de cliente, ni a los términos y condiciones legales de la empresa de transporte que utilizaste.

<sub>47 tokens de entrada · 1413 de salida · 30418 ms · 0.0 USD</sub>

---
### con-politica
**Esperada:** No: según el documento, las solicitudes posteriores al viaje quedan excluidas.

...

#### generation · google:gemini-3.5-flash-lite (repetición 1)

Basándome en la información proporcionada en el documento sobre el caso Air Canada de 2024, **sí te aplicarán la regla** (es decir, es posible que se te aplique), dado que estás realizando la solicitud **antes de completar el viaje**. 

La regla establece específicamente que la política no se aplica a las solicitudes presentadas *después* de completar el viaje. Por lo tanto, al encontrarte en el escenario opuesto (aún no viajas), cumples con el criterio temporal requerido para que la solicitud sea considerada bajo esta política.

<sub>102 tokens de entrada · 118 de salida · 1001 ms · 0.0003256 USD</sub>

#### comparison · google:gemma-4-26b-a4b-it (repetición 1)

Basándome exclusivamente en el texto que has proporcionado, la respuesta es: **La restricción mencionada no se te aplicaría.**

**Explicación de mi respuesta:**
Me baso en la lógica de la regla descrita: el documento establece una condición de exclusión específica: la política **no se aplica** si la solicitud se hace **después** de completar el viaje. Dado que tu situación es que **todavía no viajas** y planeas solicitarla **hoy**, no entras en la categoría de personas a las que se les niega la tarifa por haber completado ya el viaje.

**Lo que no puedo confirmar:**
*   **No puedo confirmar si te concederán la tarifa:** El texto solo indica cuándo *no* se aplica la política (después del viaje), pero no garantiza que la política se aplique automáticamente a todos los que soliciten antes de viajar. Podrían existir otros requisitos (documentación, comprobantes, etc.) que no están en el fragmento proporcionado.
*   **No puedo confirmar la totalidad de las reglas de Air Canada:** Mi respuesta se limita estrictamente a la interpretación de la frase que compartiste.

<sub>102 tokens de entrada · 984 de salida · 21274 ms · 0.0 USD</sub>

---
### dato-ausente
**Esperada:** El documento no indica el porcentaje; hace falta otra fuente.

#### generation · google:gemini-3.5-flash-lite (repetición 1)

Basándome en la información proporcionada en el documento, **no se menciona** ningún porcentaje de descuento para la tarifa por duelo. 

Por lo tanto, no puedo confirmar esta regla o dato con la información disponible.

<sub>98 tokens de entrada · 45 de salida · 780 ms · 0.0001419 USD</sub>

#### comparison · google:gemma-4-26b-a4b-it (repetición 1)

**No es posible determinar el porcentaje de descuento basándose en el documento proporcionado.**

**Explicación de mi respuesta:**
Me baso exclusivamente en el texto que incluiste en tu mensaje. El fragmento solo establece una **condición de temporalidad** para aplicar la política (la solicitud debe presentarse antes de completar el viaje). El documento no menciona cifras, valores monetarios ni porcentajes de descuento.

Tal como solicitaste, te indico que **no puedo confirmar la regla sobre el porcentaje** porque la información no está disponible en el texto suministrado.

<sub>98 tokens de entrada · 597 de salida · 13127 ms · 0.0 USD</sub>

## 4. Guardar la ejecución

La ejecución completa queda en un JSON dentro de `lab01/evidence/`, que no depende de las salidas del notebook: `cases.json` para el caso común y `my-task.json` para tus casos. Si vuelves a ejecutar, se reemplaza la ejecución anterior del mismo conjunto, con su revisión.

In [11]:
evidence_path = save_run(rows, cases, config, lab / "evidence", lab=1, settings=SETTINGS)
print("Ejecución guardada en", evidence_path)


Ejecución guardada en lab01/evidence/cases.json


## 5. Revisar cada respuesta

Ejecuta la celda siguiente: imprime un diccionario `REVIEW` con una entrada por respuesta y, encima de cada una, la respuesta esperada. Cópialo en la celda de abajo y completa cada par: `True` si la respuesta cumple o `False` si no, y un motivo en una frase (qué coincide, qué falta o qué se agregó sin respaldo). Relee las respuestas en «Leer las respuestas». Una respuesta incorrecta es tan útil como una correcta si explicas por qué.

In [12]:
print(review_template(evidence_path))


# Cómo completar: cada entrada es una respuesta del modelo.
#   A la izquierda, qué respuesta es: (perfil, caso, repetición). No la cambies.
#   A la derecha, tu revisión: (cumple, motivo).
#     cumple: True si la respuesta cumple la esperada o False si no; sin comillas y con mayúscula inicial.
#     motivo: una frase entre comillas que explique por qué.
#   Ejemplo: ('generation', 'con-politica', 1): (True, 'Dice que no y cita la regla del documento'),
REVIEW = {
    # Esperada: No afirma una regla: indica que necesita la política de la aerolínea para confirmarlo.
    ('generation', 'sin-politica', 1): (None, ''),
    # Esperada: No: según el documento, las solicitudes posteriores al viaje quedan excluidas.
    ('generation', 'con-politica', 1): (None, ''),
    # Esperada: El documento solo excluye las solicitudes posteriores al viaje; no confirma que se aplique antes ni sus requisitos.
    ('generation', 'antes-del-viaje', 1): (None, ''),
    # Esperada: El documento no indica el po

In [13]:
# Pega aquí el diccionario REVIEW y complétalo; puedes corregirlo y volver a ejecutar.
REVIEW = {}

evidence = apply_review(evidence_path, REVIEW, config)
print("Revisión completa:", evidence["review_complete"])


Revisión completa: False


Con la revisión completa, la celda siguiente resume cada perfil.

In [14]:
numeric = ["input_tokens", "output_tokens", "latency_ms", "estimated_cost_usd"]
reviewed = pd.DataFrame(evidence["results"]).reindex(columns=["profile", "status", "passed", *numeric])
reviewed[numeric] = reviewed[numeric].apply(pd.to_numeric, errors="coerce")
reviewed.groupby("profile").agg(
    responses=("status", lambda s: (s == "ok").sum()),
    passed=("passed", lambda s: (s == True).sum()),  # noqa: E712
    input_tokens=("input_tokens", "mean"),
    output_tokens=("output_tokens", "mean"),
    latency_ms=("latency_ms", "median"),
    cost_usd=("estimated_cost_usd", lambda s: s.sum(min_count=1)),
)


,responses,passed,input_tokens,output_tokens,latency_ms,cost_usd
profile,,,,,,
comparison,4,0,87.25,918.75,17990.5,0.000000
generation,4,0,87.25,134.25,959.0,0.001447


Lee la tabla con cuidado: con pocos casos no se puede concluir qué modelo es mejor; es una primera medición. El costo es una estimación: la fuente de cada precio queda en el JSON, y en el nivel gratuito de Gemini no pagas el precio de lista que se muestra.

## 6. Entregar y explicar

La celda siguiente comprime `lab01` con la ejecución y la revisión, y lo descarga. Descarga también este notebook con sus salidas (**Archivo → Descargar → Descargar .ipynb**). Los archivos de Colab se borran al cerrar la sesión: descarga antes de salir.

En la tarea del laboratorio 1 en Canvas, sube ambos archivos y explica en pocas líneas un caso que falló o un costo que te sorprendió, y qué cambiarías. Si usaste un asistente, declara su uso según las reglas del curso. Si ya tienes GitHub, también puedes guardar estos archivos en un repositorio propio; no es obligatorio.

Las parejas se forman después de esta clase; tu tarea y tus casos te servirán para proponer el tema del proyecto.

In [15]:
import shutil

archive = shutil.make_archive("lab01-submission", "zip", lab.parent, lab.name)
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import files
    files.download(archive)
print("Archivo para Canvas:", archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo para Canvas: /content/lab01-submission.zip
